# RNN (GRU) LTV-предсказание (v4)

Исправления vs v3:
- **Увеличение окна памяти:** `SEQ_LEN = 180` (полгода) вместо 120. Модель видит больше истории.
- **Новая архитектура фолдов:** `N_FOLDS = 2` и `STRIDE_DAYS = 14`. Теперь параметры идеально синхронизированы с чемпионом CatBoost v4a.
- **Ортогональность:** Используется как идеальный регуляризатор для бустингов (корреляция ~0.93).

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import mean_squared_error
from pathlib import Path
from datetime import date, timedelta
from tqdm.auto import tqdm
from typing import Optional

DATA_DIR = Path("../data/raw")
FEATURES_DIR = Path("../data/processed/features_v3")
MODELS_DIR = Path("../models")
SEQ_LEN = 180
BATCH_SIZE = 512
N_FOLDS = 2
STRIDE_DAYS = 14
EPOCHS = 15
PATIENCE = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

SEQ_COLS = [
    "searches", "to_cart", "to_ord", "gmv",
    "search_to_cart", "search_to_ord",
    "cat_to_cart", "cat_to_ord",
    "gmv_cat", "cat",
]

LOG_TRANSFORM_COLS = {"gmv", "searches", "gmv_cat"}

def rmsle_score(y_true, y_pred):
    y_pred_clipped = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred_clipped)))

def gini_normalized(y_true, y_pred):
    def _gini(actual, predicted):
        n = len(actual)
        indices = np.argsort(-predicted)
        sorted_actual = actual[indices]
        cumulative = np.cumsum(sorted_actual)
        gini_sum = cumulative.sum() / sorted_actual.sum() - (n + 1) / 2
        return gini_sum / n
    return _gini(y_true, y_pred) / _gini(y_true, y_true)

class LTV_GRU(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=192, num_layers=3, dropout=0.3):
        super(LTV_GRU, self).__init__()
        self.gru = nn.GRU(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, 
            batch_first=True, dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        gru_out, hidden = self.gru(x)
        return self.fc(hidden[-1]).squeeze(-1)

In [ ]:
def generate_cv_anchor_dates(
    data: pl.DataFrame,
    prediction_horizon_days: int = 30,
    stride_days: int = STRIDE_DAYS,
    min_history_days: int = 180,
    n_folds: Optional[int] = None,
) -> list[date]:
    min_date = data["event_date"].min()
    max_date = data["event_date"].max()
    latest_anchor = max_date - timedelta(days=prediction_horizon_days)
    earliest_anchor = min_date + timedelta(days=min_history_days - 1)
    n_steps = (latest_anchor - earliest_anchor).days // stride_days
    all_anchors = sorted([latest_anchor - timedelta(days=i * stride_days) for i in range(n_steps + 1)])
    return all_anchors[-n_folds:] if n_folds else all_anchors

In [ ]:
class LTVSequenceDataset(Dataset):
    def __init__(self, df_features: pl.DataFrame, df_targets: pl.DataFrame = None, is_test=False):
        self.user_ids = df_features["user_id"].to_numpy()
        self.is_test = is_test
        
        features_list = []
        for col in SEQ_COLS:
            seqs = df_features[col].to_list()
            padded = np.zeros((len(seqs), SEQ_LEN), dtype=np.float32)
            for i, s in enumerate(seqs):
                if s is None:
                    continue
                s_arr = np.atleast_1d(np.array(s, dtype=np.float32))
                if col in LOG_TRANSFORM_COLS:
                    s_arr = np.log1p(s_arr)
                length = min(len(s_arr), SEQ_LEN)
                if length > 0:
                    padded[i, -length:] = s_arr[-length:]
            features_list.append(padded)
            
        self.X = np.stack(features_list, axis=-1)
        
        if not self.is_test and df_targets is not None:
            targets = df_targets["target"].to_numpy().astype(np.float32)
            self.y = np.log1p(np.clip(targets, 0, None))
        else:
            self.y = np.zeros(len(self.user_ids), dtype=np.float32)

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])

def prepare_sequence_data(data: pl.DataFrame, anchor_date: date, user_ids: list):
    start_date = anchor_date - timedelta(days=SEQ_LEN)
    history = data.filter(
        pl.col("user_id").is_in(user_ids) & 
        (pl.col("event_date") <= anchor_date) & 
        (pl.col("event_date") > start_date)
    )
    seq_df = (
        history.sort(["user_id", "event_date"])
        .group_by("user_id", maintain_order=True)
        .agg([pl.col(c).alias(c) for c in SEQ_COLS])
    )
    return pl.DataFrame({"user_id": user_ids}).join(seq_df, on="user_id", how="left")

In [ ]:
def train_nn_fold(model, train_loader, val_loader, epochs=EPOCHS, lr=2e-3, patience=PATIENCE):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    criterion = nn.MSELoss()
    
    best_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        scheduler.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                preds = model(X_batch)
                val_loss += criterion(preds, y_batch).item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        val_rmsle = np.sqrt(val_loss)
        
        print(f"  Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val RMSLE: {val_rmsle:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        if val_rmsle < best_loss:
            best_loss = val_rmsle
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping (patience={patience})")
                break
            
    model.load_state_dict(best_model_state)
    return model, best_loss

In [ ]:
print("Чтение данных...")
raw_data = pl.read_parquet(DATA_DIR / 'train.parquet')

anchors_time_folds = generate_cv_anchor_dates(raw_data, n_folds=N_FOLDS)
anchor_end_of_time = raw_data["event_date"].max()

def load_targets(fold_path: Path) -> pl.DataFrame:
    return pl.read_parquet(fold_path / "batch_*.parquet").select(["user_id", "target"])

print("Подготовка тестового датасета...")
test_targets_df = load_targets(FEATURES_DIR / "fold_test")
test_users = test_targets_df["user_id"].to_list()
test_seq_df = prepare_sequence_data(raw_data, anchor_end_of_time, test_users)
test_dataset = LTVSequenceDataset(test_seq_df, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_predictions = np.zeros(len(test_users))
fold_scores = []

print(f"\nCumulative Walk-Forward RNN (stride={STRIDE_DAYS}, {N_FOLDS} фолдов)...")

for val_idx in range(1, N_FOLDS):
    anchor_val = anchors_time_folds[val_idx]
    train_str = "+".join([str(i) for i in range(val_idx)])
    print(f"\n--- Train: [{train_str}] | Val: fold {val_idx} ({anchor_val}) ---")

    train_datasets = []
    for train_idx in range(val_idx):
        anchor_train = anchors_time_folds[train_idx]
        train_targets = load_targets(FEATURES_DIR / f"fold_{train_idx:02d}")
        train_users = train_targets["user_id"].to_list()
        train_seq_df = prepare_sequence_data(raw_data, anchor_train, train_users)
        train_datasets.append(LTVSequenceDataset(train_seq_df, train_targets))
    
    combined_train = ConcatDataset(train_datasets)
    
    val_targets = load_targets(FEATURES_DIR / f"fold_{val_idx:02d}")
    val_users = val_targets["user_id"].to_list()
    val_seq_df = prepare_sequence_data(raw_data, anchor_val, val_users)
    val_dataset = LTVSequenceDataset(val_seq_df, val_targets)
    
    print(f"  Train: {len(combined_train):,} | Val: {len(val_dataset):,}")
    
    train_loader = DataLoader(combined_train, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = LTV_GRU(input_dim=len(SEQ_COLS), hidden_dim=192, num_layers=3, dropout=0.3).to(DEVICE)
    model, best_val = train_nn_fold(model, train_loader, val_loader)
    fold_scores.append(best_val)

    torch.save(model.state_dict(), MODELS_DIR / f"gru_v3_fold_{val_idx}.pth")

    model.eval()
    fold_preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            preds = model(X_batch.to(DEVICE))
            fold_preds.extend(preds.cpu().numpy())
    test_predictions += np.expm1(np.clip(fold_preds, 0, None)) / (N_FOLDS - 1)

print(f"\nСреднее Val RMSLE: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

submit_df = pd.DataFrame({"user_id": test_users, "predict": np.clip(test_predictions, 0, None)})
submit_path = Path("../data/processed/rnn_v4_submission.csv")
submit_df.to_csv(submit_path, index=False)
print(f"Сохранено: {submit_path}")
print(f"mean={submit_df['predict'].mean():.2f} | median={submit_df['predict'].median():.2f}")